In [7]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F
from torch.nn import Linear, ModuleList
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric.nn import Set2Set

import wandb




## 2. Get the dataset

The dataset used is `combined_dataset.csv`


In [9]:
# load dataset
DATA_ROOT = './data/models'
RAW_DIR  = os.path.join(DATA_ROOT, 'raw')

df = pd.read_csv(os.path.join(RAW_DIR, 'combined_dataset.csv'), index_col="SEQID")

df.sort_index(inplace=True)

with open(os.path.join(RAW_DIR, 'combined_data_split.json'), 'r') as f:
    split_dict = json.load(f)

print(f"Total records: {len(df)}")
print(df['dataset'].value_counts())
df.head()


Total records: 30872
arr       27730
ov         2775
lit_uv      348
uv           19
Name: dataset, dtype: int64


,dataset,RefSeq,TargetStruct,sodium,DNA_conc,dH,Tm,dG_37
SEQID,,,,,,,,
BC1,arr,CAACCAGAAATGGTTG,((((((....)))))),0.063,NaN,-29.405457,52.382626,-1.389517
BC10,arr,CAATCAGAAATGGTTG,(((.((....)).))),0.063,NaN,-13.860604,28.607763,0.385480
BC11,arr,CAAGCAGAAATGGTTG,(((.((....)).))),0.063,NaN,-19.310492,32.871813,0.260496
BC15,arr,AGAAACCGAAAGGCTTCT,((((.((....)).)))),0.063,NaN,-28.359914,40.193048,-0.288995
BC16,arr,AGAATCCGAAAGGCTTCT,((((.((....)).)))),0.063,NaN,-27.266132,34.866015,0.188904


python: can't open file '/home/anant/nnn_paper/dict': [Errno 2] No such file or directory


## 2. Converting each Record to a graph notation

In the dataset we have below two parameters for each sequence-
1. RefSeq e.g. `ATGCGCTA`
2. TargetStruct e.g. `(((....)))`

The above two are parameters for representing the sequence/dna hairpin in a graph format.


Now a graph has:
1. Nodes : one node for each nucleotide i.e. `A`, `T`, `G` and `C`
2. Edges : There are 3 types of edges.
    1. $5'-3'$ edge representing a froward link
    2. $3'-5'$ edge representing a backward link
    3. hydrogen bond <br>
    
The bond `1` and `2` are classified as backbone bond.<br>`.` in the `TargetStruct` represent a hydrogen bond. 


In [28]:
# ---- Feature encoding functions ----

def onehot_nucleotide(seq_str: str) -> np.ndarray:
    """One-hot encode a DNA sequence. Returns (N, 4) array."""
    mapping = {'A': 0, 'T': 1, 'C': 2, 'G': 3}
    N = len(seq_str)
    arr = np.zeros((N, 4), dtype=np.float32)
    for i, nt in enumerate(seq_str.upper()):
        arr[i, mapping[nt]] = 1.0
    return arr


def dotbracket_to_edges(struct: str):
    """
    Convert dot-bracket notation to edge list + edge features.
    
    Returns:
        edge_index: (2, num_edges) tensor
        edge_attr: (num_edges, 3) tensor - [is_5to3, is_3to5, is_hbond]
    """
    N = len(struct)
    strand_break = struct.find('+')
    
    if strand_break == -1:
        # Hairpin: backbone connects all consecutive nucleotides
        backbone = [[i, i+1] for i in range(N - 1)]
    else:
        # Duplex: remove '+' and skip the break position
        struct = struct.replace('+', '')
        N -= 1
        backbone = [[i, i+1] for i in range(N - 1)
                    if i != strand_break - 1 and i + 1 != strand_break]
    
    # Find hydrogen bonds from matched parentheses
    hbonds = []
    flag3p = N - 1
    for i, ch in enumerate(struct):
        if ch == '(':
            for j in range(flag3p, i, -1):
                if struct[j] == ')':
                    hbonds.append([i, j])
                    flag3p = j - 1
                    break
    
    # Build full edge list: 5'→3', 3'→5', hbond (both directions)
    print(f"Backbone {backbone}\nBackbone reversed {[e[::-1] for e in backbone]}\nHbond {hbonds}\nHbonds rev {[e[::-1] for e in hbonds]}")
    print(f"shape of each\nBackbone {torch.tensor(backbone).shape}\nHbond {torch.tensor(hbonds).shape}")
    edges = (backbone + 
             [e[::-1] for e in backbone] + 
             hbonds + 
             [e[::-1] for e in hbonds])
    

    print(edges)
    print(f"Shape of edges is {torch.tensor(edges).shape}")
    n_bb, n_hb = len(backbone), len(hbonds)
    edge_attr = np.zeros((len(edges), 3), dtype=np.float32)
    edge_attr[:n_bb, 0] = 1           # 5' → 3'
    edge_attr[n_bb:2*n_bb, 1] = 1     # 3' → 5'
    edge_attr[2*n_bb:, 2] = 1         # hydrogen bonds
    
    edge_index = torch.tensor(np.array(edges).T, dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    return edge_index, edge_attr


# Quick sanity check
ei, ea = dotbracket_to_edges('((..))')
print(f'Structure "((..))": {ei.shape[1]} edges, {ea.shape}')
print(f'One-hot "ATCG":\n{onehot_nucleotide("ATCGATGC")}')

Backbone [[0, 1], [1, 2], [2, 3], [3, 4], [4, 5]]
Backbone reversed [[1, 0], [2, 1], [3, 2], [4, 3], [5, 4]]
Hbond [[0, 5], [1, 4]]
Hbonds rev [[5, 0], [4, 1]]
shape of each
Backbone torch.Size([5, 2])
Hbond torch.Size([2, 2])
[[0, 1], [1, 2], [2, 3], [3, 4], [4, 5], [1, 0], [2, 1], [3, 2], [4, 3], [5, 4], [0, 5], [1, 4], [5, 0], [4, 1]]
Shape of edges is torch.Size([14, 2])
Structure "((..))": 14 edges, torch.Size([14, 3])
One-hot "ATCG":
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]


In [17]:
def calc_sumstats(df):
    """Compute min/max for dH and Tm from a dataframe. """
    """this function assums the input does not have NO nan values. """
    return {
        'dH_min' : np.min(df.dH), 'dH_max': np.max(df.dH),
        'Tm_min': np.min(df.Tm), 'Tm_max': np.max(df.Tm)
    } 

def normalize(val, vmin, vmax):
    return (val - vmin) / (vmax - vmin)

def unnormalize(val, vmin, vmax):
    return val * (vmax - vmin) + vmin

train_df = df.loc[df.index.isin(split_dict['train_ind'])]

stats = calc_sumstats(train_df)
print('Training set normalization stats:')
stats

Training set normalization stats:


{'dH_min': -68.20454632608892,
 'dH_max': -2.70243642577065,
 'Tm_min': 13.647488646379209,
 'Tm_max': 68.63588396988382}


## 3. Understanding `np.nanmin`

`np.nanmin` is a NumPy function that returns the minimum value of an array while **ignoring NaN (Not a Number) values**.

- **Standard `np.min`**: Returns `nan` if any NaN values are present in the array
- **`np.nanmin`**: Ignores NaN values and returns the minimum of the remaining valid numbers

This is particularly useful when working with datasets that contain missing values (represented as NaN), as it allows you to compute the minimum without having to preprocess or filter out the NaN values first.

**Example:**
```python
import numpy as np

arr = np.array([1, 2, np.nan, 4, 5])
print(np.min(arr))     # Output: nan
print(np.nanmin(arr))  # Output: 1.0
```
## 4. Understanding `isinstance(refseq, list)` and `''.join(refseq)`

This code snippet handles the case where `refseq` (the reference sequence) might be stored as a list instead of a string.

### `isinstance(refseq, list)`
- Checks if the variable `refseq` is of type `list`
- Returns `True` if `refseq` is a list, `False` otherwise
- This is useful for handling different data formats in the dataset

### `''.join(refseq)`
- Joins all elements of the list into a single string
- The empty string `''` is the separator (no characters between elements)
- Converts a list like `['ATGC', 'CGTA']` into the string `'ATGCCGTA'`

**Example:**
```python
# If refseq is a list
refseq = ['ATGC', 'CGTA']
refseq = ''.join(refseq)
print(refseq)  # Output: 'ATGCCGTA'

# If refseq is already a string
refseq = 'ATGCCGTA'
# isinstance check returns False, no conversion needed
```

**Why is this needed?**

In the dataset, some sequences (particularly duplexes) may be stored as lists to represent separate strands, while single-stranded hairpins are stored as strings. This code ensures consistent string format for downstream processing, regardless of the original storage format.


# Understanding `''.join(eval(refseq))`

This code snippet converts a string representation of a list into an actual list, then joins it into a single string.

## Breaking it down:

1. **`eval(refseq)`**: Evaluates the string as Python code
    - If `refseq = "['ATGC', 'CGTA']"` (a string that looks like a list)
    - `eval(refseq)` converts it to the actual list `['ATGC', 'CGTA']`
    - ⚠️ **Security warning**: `eval()` can execute arbitrary code, so only use it on trusted data

2. **`''.join(...)`**: Joins the list elements with no separator
    - Takes `['ATGC', 'CGTA']` and produces `'ATGCCGTA'`

## Example:

```python
refseq = "['AT', 'GC']"  # String representation of a list
result = ''.join(eval(refseq))
print(result)  # Output: 'ATGC'
```

## Why use it?

In your dataset, some sequences are stored as string representations of lists (e.g., `"['strand1', 'strand2']"` for duplex sequences) rather than actual strings or list objects. This code handles that specific storage format by converting it back to a plain sequence string.

**Note**: This is different from the earlier approach using `isinstance()` which checks if the variable is already an actual list object.

In [ ]:
def row_to_graph(row, stats):
    """
    Converts a single dataframe row into troch_geometric Data object.mro
    Args:
        row: pandas Series with RefSeq, TargetStruct, dH, Tm
        stats: dict with min and max for dH and Tm
    Returns:
        Data(x, edge_inded, edge_attr, y)
    """
    # Parse RefSeq (handle string-repr lists for duplex)
    reference_sequence = row['RefSeq']
    if isinstance(reference_sequence, list):
        reference_sequence = ''.join(reference_sequence)
    elif isinstance(reference_sequence, str) and '[' in reference_sequence:
        reference_sequence = ''.join(eval(reference_sequence))

    # Node features: one-hot nucleotides
    x = torch.tensor(onehot_nucleotide(reference_sequence), dtype=torch.float)

    # Edge features: from dot-bracket structure
    edge_index, edge_attr = dotbracket_to_edges(row['TargetStruct'])

    # Normalize the labels

    dH_norm = normalize(row.dH, stats['dH_min'], stats['dH_max'])
    Tm_norm = normalize(row.Tm, stats['Tm_min'], stats['Tm_max'])
    y = torch.tensor([dH_norm, Tm_norm], dtype=torch.float)

    return Data(x = x, edge_index=edge_index, edge_attr=edge_attr, y=y)

sample = df.iloc[0]
g = row_to_graph(sample, stats)
print(f"Sample graph: {g.num_nodes} nodes, {g.num_edges} edges, y={g.y}")

Sample graph: 16 nodes, 42 edges, y=tensor([0.5923, 0.7044])


In [ ]:
class NNNGraphDataset(InMemoryDataset):
    """
    Converts combined_dataset.csv into graph dataset
    Processes once, then loads from disk.
    """

    def __init__(self, root, transform=None):
        super().__init__(root, transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

        # Load metadata

IndentationError: expected an indented block (628279208.py, line 7)